In [152]:
import cohere
import os
import numpy as np
import pandas as pd
from numpy.typing import NDArray

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
co = cohere.ClientV2(COHERE_API_KEY)


# Import all data from the scifact
df_train = pd.read_csv("./data/scifact/qrels/train.tsv", sep='\t')
df_test = pd.read_csv("./data/scifact/qrels/test.tsv", sep='\t')
df_corpus = pd.read_json("./data/scifact/corpus.jsonl", lines=True)
df_queries = pd.read_json("./data/scifact/queries.jsonl", lines=True)

display(df_train.info())
# display(df_test.info())
display(df_corpus.info())
display(df_queries.info())

display(df_corpus.head())
display(df_queries.head())

# def calc_cosine_similarity(doc1, doc2):
#     return np.dot(doc1,doc2) / ( np.linalg.norm(doc1) * np.linalg.norm(doc2))

def calculate_vectorized_similarity(query_embed: NDArray, corpus_embed: NDArray):
    return (query_embed @ corpus_embed.T / (np.linalg.norm(query_embed, axis=-1, keepdims=True) * np.linalg.norm(corpus_embed, axis=-1).reshape(1,-1)))

<class 'pandas.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   query-id   919 non-null    int64
 1   corpus-id  919 non-null    int64
 2   score      919 non-null    int64
dtypes: int64(3)
memory usage: 21.7 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 5183 entries, 0 to 5182
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   _id       5183 non-null   int64 
 1   title     5183 non-null   str   
 2   text      5183 non-null   str   
 3   metadata  5183 non-null   object
dtypes: int64(1), object(1), str(2)
memory usage: 7.6+ MB


None

<class 'pandas.DataFrame'>
RangeIndex: 1109 entries, 0 to 1108
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   _id       1109 non-null   int64 
 1   text      1109 non-null   str   
 2   metadata  1109 non-null   object
dtypes: int64(1), object(1), str(1)
memory usage: 122.6+ KB


None

,_id,title,text,metadata
0,4983,Microstructural development of human newborn c...,Alterations of the architecture of cerebral wh...,{}
1,5836,Induction of myelodysplasia by myeloid-derived...,Myelodysplastic syndromes (MDS) are age-depend...,{}
2,7912,"BC1 RNA, the transcript from a master gene for...",ID elements are short interspersed elements (S...,{}
3,18670,The DNA Methylome of Human Peripheral Blood Mo...,DNA methylation plays an important role in bio...,{}
4,19238,The human myelin basic protein gene is include...,Two human Golli (for gene expressed in the oli...,{}


,_id,text,metadata
0,0,0-dimensional biomaterials lack inductive prop...,{}
1,2,1 in 5 million in UK have abnormal PrP positiv...,"{'13734012': [{'sentences': [4], 'label': 'CON..."
2,4,1-1% of colorectal cancer patients are diagnos...,{}
3,6,10% of sudden infant death syndrome (SIDS) dea...,{}
4,9,32% of liver transplantation programs required...,"{'44265107': [{'sentences': [15], 'label': 'SU..."


# Baseline comparison before Cohere Reranker


In [153]:
from cohere import EmbedInputType 
import time
import math

PRODUCTION = True

class MockResponse():
    embeddings = []
    def __init__(self, array):
        self.embeddings = array

def load_sorted_checkpoint(filename):
    with np.load(filename) as archive:
        ids = archive['ids']
        embeddings = archive['embeddings']
    if len(ids) != len(embeddings):
        raise ValueError('Checkpoint IDs and embedding rows must have equal lengths')
    order = np.argsort(ids, kind='stable')
    ids, embeddings = ids[order], embeddings[order]
    np.savez(filename, ids=ids, embeddings=embeddings)
    return ids, embeddings

def embed_cohere_batched(model, input_type: EmbedInputType, text_or_corpus: pd.DataFrame,  max_texts_per_call = 96, filename = ""):
    filename_npz = filename.replace(".npy", ".npz")

    
    # Check how many files we have left
    try:
        cached_ids, cached_embeddings = load_sorted_checkpoint(filename_npz)
        text_or_corpus = text_or_corpus[~text_or_corpus['_id'].isin(cached_ids)] 

        # Early cache exit
        if len(text_or_corpus) == 0:
            return cached_embeddings
    except FileNotFoundError:
        pass
    
    # embeds = [] now we save in a file
    TOTAL_CHUNKS = math.ceil(len(text_or_corpus) / max_texts_per_call) 

    for n_call in range(1,TOTAL_CHUNKS + 1):
        chunk = text_or_corpus.iloc[(n_call-1) * max_texts_per_call :n_call * max_texts_per_call,]
        # print(n_call, len(chunk))

        try:
            checkpoint_embedded_array = np.load(filename_npz)
            if all(np.isin(checkpoint_embedded_array['ids'],list(chunk['_id']))):
                continue
        except:
            None

        if PRODUCTION:
            for attempt in range(6):
                try:
                    cohere_embed_response = co.embed(model=model,
                                            input_type=input_type,
                                            texts=list(chunk['text']),
                                            embedding_types=['float'])
                    # Save while batching to avoid issues... with index
                    if filename_npz:
                        checkpoint_embedded_array = np.array(cohere_embed_response.embeddings.float_)
                        ids = chunk['_id'].to_numpy()
                        try:
                            with np.load(filename_npz) as archive:
                                checkpoint_embedded_array = np.concatenate(
                                    (archive['embeddings'], checkpoint_embedded_array), axis=0)
                                ids = np.concatenate((archive['ids'], ids))
                        except FileNotFoundError:
                            pass

                        order = np.argsort(ids, kind='stable')
                        ids = ids[order]
                        checkpoint_embedded_array = checkpoint_embedded_array[order]

                        np.savez(filename_npz, 
                                 ids=ids,
                                 embeddings=checkpoint_embedded_array)

                    break

                except cohere.TooManyRequestsError:
                    if attempt == 5:
                        raise
                    print(f'Batch {n_call} rate limited; retrying the same batch in 60 seconds')
                    time.sleep(60)

        else:
            cohere_embed_response = MockResponse([np.random.normal(0,1,1024) for x in range(max_texts_per_call)])
        # embeds += cohere_embed_response.embeddings
        # print(np.random.normal(0,1,1024))
        # break;
    
    return load_sorted_checkpoint(filename_npz)[1]



# do embedding
embed_models = [
    # embed-v5.0-pro
    # Our most capable embedding model, for text and images. Higher quality than embed-v5.0-fast, at higher latency.
    # Text, Images, Mixed texts/images (i.e. PDFs)
    # One of ‘[256, 512, 768, 1024, 1536, 2048 (default)]‘
    # 128k
    # Cosine Similarity, Dot Product Similarity, Euclidean Distance
    # Embed
    'embed-v5.0-pro',
    # embed-v5.0-fast
    # A faster, lighter version of embed-v5.0-pro. Allows for text and images to be classified or turned into embeddings
    # Text, Images, Mixed texts/images (i.e. PDFs)
    # One of ‘[256, 512, 768, 1024, 1536, 2048 (default)]‘
    # 128k
    # Cosine Similarity, Dot Product Similarity, Euclidean Distance
    # Embed
    'embed-v5.0-fast'
]
model = embed_models[-1]

# saving locally since it's a small dataset
EMBED_CORPUS_FILE_NAME = model + '_cache_embedded_corpus.npy'
EMBED_QUERIES_FILE_NAME = model + '_cache_embedded_queries.npy'

# Always resume through the ID-aware .npz checkpoint.
# Legacy .npy files may contain mock vectors and are not loaded.
# embedded_corpus = np.array([])
# embedded_queries = np.array([])

# print(embed_cohere_batched(model,"search_document", list(df_corpus['text'])))
# print(embedded_corpus.shape)
# print(embedded_queries.shape)

embedded_corpus = embed_cohere_batched(model,"search_document", df_corpus[['_id','text']],filename=EMBED_CORPUS_FILE_NAME)
print(embedded_corpus.shape)
if len(embedded_corpus):
    np.save(EMBED_CORPUS_FILE_NAME, np.array(embedded_corpus))

embedded_queries = embed_cohere_batched(model,"search_query", df_queries[['_id','text']],filename=EMBED_QUERIES_FILE_NAME)
print(embedded_queries.shape)                            
# cache with numpy the results of all the embeds...
if len(embedded_queries):
    np.save(EMBED_QUERIES_FILE_NAME, np.array(embedded_queries))



(5183, 2048)
(1109, 2048)


In [154]:
df_train = df_train.set_index("query-id").groupby(level=0)
df_corpus = df_corpus.set_index('_id')

In [155]:
df_train.head()

,corpus-id,score
query-id,,
0,31715818,1
2,13734012,1
4,22942787,1
6,2613775,1
9,44265107,1
...,...,...
1404,38355793,1
1405,10504681,1
1406,2617858,1


In [158]:
result_corpus = np.load(EMBED_CORPUS_FILE_NAME.replace(".npy",".npz"))
result_queries = np.load(EMBED_QUERIES_FILE_NAME.replace(".npy",".npz"))
print(len(result_corpus['ids']), len(result_corpus['embeddings']))

# print(embedded_queries.shape, embedded_corpus.T.shape)
# auto transposes
#Should be (Q,C)
similarity_results = calculate_vectorized_similarity(embedded_queries,embedded_corpus)

print(similarity_results.shape)

args_sorted_similarity_q_c = np.argsort(-similarity_results, axis=1)

# print(args_sorted_similarity_q_c)

# with np.load(EMBED_QUERIES_FILE_NAME.replace('.npy', '.npz')) as result_queries:
query_id = result_queries['ids'][0]
# print(df_queries.loc[df_queries['_id'] == query_id, 'text'])

k=3

def get_k_results(sim_results, k):
    k_results = sim_results[:,:k]
    print("kres", k_results)
    avg_k = 0
    count = 0
    for q_index, documents in enumerate(k_results):
        query_id = result_queries['ids'][q_index]
    
        if query_id not in df_train.groups:
            continue
        # df_queries.loc[df_queries['_id'] == query_id]
        group = df_train.get_group(query_id)

        query_corpus = group['corpus-id']
        # print("query_corpus",query_corpus)
        total_relevant = query_corpus.count()
        result = query_corpus.isin([ result_corpus['ids'][doc] for doc in documents]).sum()
        print(result, result / total_relevant)
        avg_k += result / total_relevant
        count += 1

    return avg_k / count

print("K results avg", get_k_results(args_sorted_similarity_q_c, k))

# for c_index, q_index_result in enumerate(args_sorted_similarity_q_c[0][:k]):
#     # print(c_index, )
#     print(c_index, q_index_result)#embedded_queries[0] 
#     print(df_corpus.loc[result_corpus['ids'][q_index_result],'text'])
#     print(similarity_results[0][q_index_result] * 100)

5183 5183
(1109, 5183)
kres [[ 766 2723 3190]
 [ 766 2723 3190]
 [2232 2862 3657]
 ...
 [1789 4630 2456]
 [ 488  218  316]
 [1367 1882 2810]]
0 0.0
1 1.0
0 0.0
1 1.0
1 1.0
0 0.0
0 0.0
1 1.0
0 0.0
0 0.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
0 0.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
2 1.0
1 1.0
1 1.0
1 1.0
1 0.5
1 0.5
0 0.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
0 0.0
1 1.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
1 1.0
2 1.0
2 1.0
2 1.0
0 0.0
0 0.0
0 0.0
0 0.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
2 1.0
3 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
1 1.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
3 0.75
1 1.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
2 1.0
1 1.0
1 1.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
0 0.0
0 0.0
0 0.0
1 1.0
1 1.0
1 1.0
1 1.0
2 1.0
2 1.0
1 1.0
1 1.0
1 0.5

In [ ]:
df_train = df_train.set_index('query-id')

AttributeError: 'DataFrameGroupBy' object has no attribute 'set_index'

In [ ]:
# Lets find the id of the first query, and the relevant documents

df_train.head()


# df_train.loc[0,'corpus-id']['score'] == 1
# for q in 
df_train[df_corpus]
#[df_train['query-id'] == 0]['corpus-id'] == 

,corpus-id,score
query-id,,
0,NaN,NaN
2,NaN,NaN
4,NaN,NaN
6,NaN,NaN
9,NaN,NaN
...,...,...
1404,NaN,NaN
1405,NaN,NaN
1406,NaN,NaN


# Cohere Reranker

In [ ]:
from cohere import V2RerankResponse
# rerank
rerank_models = [
    # 10k docs max
    'rerank-v5.0-pro',
    'rerank-v5.0-fast'
]
q = ""
docs = []
ks = [1, 3, 5, 10, 100]

def validate_result(response:V2RerankResponse):
    return response

for k in ks:
    response = co.rerank(model=rerank_models[-1],
                         query=q,
                         documents=docs,
                         top_n=k
                         )
    break


BadRequestError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '4e6e39a8cab2aa0a7a6382c1eaeb36a6', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '10', 'x-trial-endpoint-call-remaining': '9', 'date': 'Tue, 27 Oct 2026 22:33:17 GMT', 'x-envoy-upstream-service-time': '15', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 400, body: {'id': 'c981140c-9734-4eb6-9152-68136afd9f14', 'message': 'invalid request: list of documents must not be empty'}

# Results Local Vs Cohere Reranker vs Others
